# 📊 Screener de Ações — Ibovespa

**Objetivo:** Identificar as 20 melhores oportunidades do Ibovespa (ações + bancos) que atendam a critérios fundamentalistas rigorosos e estejam com preço de mercado abaixo do preço justo.

**Critérios de Screening (Ações — 11 critérios):**
| # | Critério | Valor |
|---|----------|-------|
| 1 | P/L | entre 0 e 10 |
| 2 | P/PV | entre 0 e 1,5 |
| 3 | Margem EBIT | positiva |
| 4 | Margem Líquida | > 10% |
| 5 | Dívida Líq./EBIT | < 3 |
| 6 | Dívida Líq./PL | < 2 |
| 7 | ROE | > 10% |
| 8 | Liquidez Corrente | > 1 |
| 9 | Passivos/Ativos | < 1 |
| 10 | Liq. Média Diária | > R$ 100 mil |
| 11 | LPA | > 0 |

**Critérios de Screening (Bancos — 7 critérios):**
P/L 0–10, P/PV 0–2, ROE > 15%, Margem Líq. > 10%, LPA > 0, Liq. Diária > R$ 100k, DY > 3%.

**Valuation (inspirado Simply Wall St):**
- **Ações:** DCF 2-estágios (decay linear, 10 anos) + DDM fallback + Graham
- **Bancos:** Excess Returns Model + Graham
- ⭐ **Forte desconto** = margem de segurança média ≥ 20%

In [1]:
# === Setup ===
import sys
import importlib
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 200)

# Garantir que o diretório raiz está no path para importar src/
if '.' not in sys.path:
    sys.path.insert(0, '.')

from src import scraper, fundamentals, filters, valuation

# Recarregar módulos durante desenvolvimento
for mod in [scraper, fundamentals, filters, valuation]:
    importlib.reload(mod)

print("✅ Módulos carregados com sucesso")

✅ Módulos carregados com sucesso


## 1. Coleta de Tickers

Scraping de `dadosdemercado.com.br/acoes` para obter a lista completa de ações brasileiras, seguido de classificação em **bancos** e **não-bancos**.

In [2]:
# Obter tickers de ações brasileiras
tickers_df = scraper.get_tickers()
print(f"Total de tickers: {len(tickers_df)}")
tickers_df.head(10)

[scraper] 384 tickers obtidos de dadosdemercado.com.br
Total de tickers: 384


,ticker,ticker_sa
0,PETR4,PETR4.SA
1,B3SA3,B3SA3.SA
2,RAIZ4,RAIZ4.SA
3,ITUB4,ITUB4.SA
4,CSAN3,CSAN3.SA
5,ITSA4,ITSA4.SA
6,LREN3,LREN3.SA
7,BBDC3,BBDC3.SA
8,MGLU3,MGLU3.SA
9,COGN3,COGN3.SA


## 2. Coleta de Dados Fundamentalistas

Busca de métricas via `yfinance` para todas as ações. Isso pode levar alguns minutos dependendo do número de tickers.

In [3]:
# Coletar fundamentals para todas as ações
stock_fundamentals = fundamentals.fetch_fundamentals(tickers_df['ticker_sa'].tolist(), delay=0.4)
stock_fundamentals.head()

Coletando fundamentals: 100%|██████████| 384/384 [14:40<00:00,  2.29s/it]


[fundamentals] 384 tickers processados, 381 com dados de preço


,ticker_sa,ticker,nome,setor,industria,preco,pl,pvp,margem_ebit_pct,margem_liquida_pct,dl_ebit,dl_pl,roe_pct,liquidez_corrente,passivos_ativos,liq_media_diaria,lpa,vpa,dy_pct,divida_liquida,ebit,fcf_latest,shares_outstanding,dividend_rate
0,PETR4.SA,PETR4,PETROBRAS PN ATZ N2,Energy,Oil & Gas Integrated,49.66,6.09,1.54,30.41,22.13,12.07,4.39,28.18,0.71,0.66,3353414656.80,8.16,32.26,7.91,333417000960.00,27614797780.70,16721236676.89,5446501379.00,3.91
1,B3SA3.SA,B3SA3,B3 ON NM,Financial Services,Financial Data & Stock Exchanges,17.07,20.57,4.93,86.09,45.54,0.13,0.06,25.59,1.91,0.64,613931085.00,0.83,3.46,3.52,1114740736.00,8666644000.00,4071357000.00,5010732147.00,0.61
2,RAIZ4.SA,RAIZ4,RAIZEN PN N2,Utilities,Utilities - Renewable,0.54,NaN,-3.51,0.22,-9.58,107.48,3.51,-231.28,1.69,0.87,31593331.80,-2.15,-0.15,NaN,61700040704.00,574045000.00,-5241328000.00,1349543297.00,NaN
3,ITUB4.SA,ITUB4,ITAUUNIBANCOPN EJ N1,Financial Services,Banks - Regional,41.39,10.32,2.23,NaN,32.28,NaN,2.86,21.01,NaN,0.93,1227145206.50,4.01,18.55,1.32,585442983936.00,NaN,27351000000.00,5403525775.00,0.55
4,CSAN3.SA,CSAN3,COSAN ON NM,Energy,Oil & Gas Refining & Marketing,5.10,NaN,3.80,0.65,-24.05,155.49,7.69,-28.96,2.58,0.77,197257392.00,-3.89,1.34,NaN,40807221248.00,262450000.00,4566917000.00,3906847275.00,NaN


## 3. Screening — Ações (não-bancos)

Aplicação dos 11 critérios fundamentalistas.

In [ ]:
# Aplicar filtros fundamentalistas para ações
# Sanitizar colunas numéricas para evitar comparação str vs int nos filtros
numeric_cols = [
    'preco', 'pl', 'pvp', 'margem_ebit_pct', 'margem_liquida_pct',
    'dl_ebit', 'dl_pl', 'roe_pct', 'liquidez_corrente', 'passivos_ativos',
    'liq_media_diaria', 'lpa', 'vpa', 'dy_pct', 'divida_liquida', 'ebit',
    'fcf_latest', 'shares_outstanding', 'dividend_rate'
]

stock_fundamentals_clean = stock_fundamentals.copy()
for col in numeric_cols:
    if col in stock_fundamentals_clean.columns:
        stock_fundamentals_clean[col] = pd.to_numeric(
            stock_fundamentals_clean[col], errors='coerce'
        )

filtered_stocks = filters.apply_stock_filters(stock_fundamentals_clean)

if len(filtered_stocks) > 0:
    display_cols = ['ticker', 'nome', 'setor', 'preco', 'pl', 'pvp',
                    'margem_ebit_pct', 'margem_liquida_pct', 'dl_ebit', 'dl_pl',
                    'roe_pct', 'liquidez_corrente', 'passivos_ativos', 'lpa']
    display(filtered_stocks[display_cols].style.format({
        'preco': 'R$ {:.2f}', 'pl': '{:.2f}', 'pvp': '{:.2f}',
        'margem_ebit_pct': '{:.1f}%', 'margem_liquida_pct': '{:.1f}%',
        'dl_ebit': '{:.2f}', 'dl_pl': '{:.2f}', 'roe_pct': '{:.1f}%',
        'liquidez_corrente': '{:.2f}', 'passivos_ativos': '{:.2f}', 'lpa': 'R$ {:.2f}',
    }))
else:
    print("⚠️ Nenhuma ação passou em todos os 11 critérios."
          "\nIsso pode ocorrer em mercados sobrevalorizados."
          "\nConsidere relaxar alguns critérios.")

NameError: name 'stock_fundamentals' is not defined

## 4. Valuation — Preço Justo

Cálculo do preço justo de cada ação filtrada por dois métodos:
1. **DCF 2-Estágios** — Crescimento decai linearmente de CAGR histórico → inflação LP (3,5%) ao longo de 10 anos. Fallback: DDM quando FCF indisponível.
2. **Fórmula de Graham** — `V = √(P/L_setor × P/PV_setor × LPA × VPA)`

Para **bancos**, o modelo primário é **Excess Returns**: `FV = VPA + (ROE - CoE) × VPA / (CoE - g)`

Ações com preço de mercado **abaixo de ambos** os preços justos são consideradas subvalorizadas.
⭐ **Forte desconto** = margem de segurança média ≥ 20% (inspirado Simply Wall St).

In [ ]:
# Calcular valuation (DCF 2-estágios + Graham) para ações filtradas
if len(filtered_stocks) > 0:
    valued_stocks = valuation.apply_valuation(filtered_stocks, stock_fundamentals_clean, model='stock')

    val_cols = ['ticker', 'nome', 'preco', 'preco_justo_dcf', 'preco_justo_graham',
                'margem_seg_dcf_pct', 'margem_seg_graham_pct', 'undervalued', 'forte_desconto']
    display(valued_stocks[val_cols].style.format({
        'preco': 'R$ {:.2f}',
        'preco_justo_dcf': 'R$ {:.2f}',
        'preco_justo_graham': 'R$ {:.2f}',
        'margem_seg_dcf_pct': '{:.1f}%',
        'margem_seg_graham_pct': '{:.1f}%',
    }))
else:
    valued_stocks = pd.DataFrame()
    print("⚠️ Nenhuma ação para avaliar.")

[valuation] 18/23 ações abaixo do preço justo | 15 com forte desconto (≥20%)


,ticker,nome,preco,preco_justo_dcf,preco_justo_graham,margem_seg_dcf_pct,margem_seg_graham_pct,undervalued,forte_desconto
0,CYRE3,CYRELA REALTON NM,R$ 26.01,R$ 35.35,R$ 34.22,26.4%,24.0%,True,True
1,CMIG4,CEMIG PN EJ N1,R$ 12.33,R$ 29.41,R$ 16.68,58.1%,26.1%,True,True
2,RECV3,PETRORECSA ON NM,R$ 13.97,R$ 62.68,R$ 19.18,77.7%,27.1%,True,True
3,JHSF3,JHSF PART ON NM,R$ 8.63,R$ 7.63,R$ 10.68,-13.1%,19.2%,False,False
4,GRND3,GRENDENE ON NM,R$ 4.62,R$ 4.79,R$ 4.77,3.5%,3.2%,True,False
5,EZTC3,EZTEC ON NM,R$ 13.35,R$ 14.64,R$ 14.76,8.8%,9.6%,True,False
6,ALUP11,ALUPAR UNT N2,R$ 34.53,R$ 51.30,R$ 43.46,32.7%,20.5%,True,True
7,SAPR4,SANEPAR PN N2,R$ 8.27,R$ 78.38,R$ 29.29,89.4%,71.8%,True,True
8,MELK3,MELNICK ON NM,R$ 3.34,R$ 4.70,R$ 4.18,29.0%,20.0%,True,True
9,EVEN3,EVEN ON NM,R$ 7.02,R$ 7.16,R$ 9.31,2.0%,24.6%,True,False


## 4.1. Screening + Valuation — Bancos

Filtros adaptados para bancos (7 critérios) + valuation via **Excess Returns Model** e **Graham**.

In [ ]:
# Separar bancos dos dados fundamentalistas usando a classificação do scraper
from src.scraper import KNOWN_BANK_TICKERS, BANK_INDUSTRIES

bank_mask = (
    stock_fundamentals_clean['ticker'].isin(KNOWN_BANK_TICKERS) |
    stock_fundamentals_clean['industria'].str.lower().isin(BANK_INDUSTRIES)
)
bank_fundamentals = stock_fundamentals_clean[bank_mask].copy()
print(f"Bancos identificados: {len(bank_fundamentals)}")

# Aplicar filtros para bancos
filtered_banks = filters.apply_bank_filters(bank_fundamentals)

# Calcular valuation (Excess Returns + Graham) para bancos
if len(filtered_banks) > 0:
    valued_banks = valuation.apply_valuation(filtered_banks, stock_fundamentals_clean, model='bank')

    val_cols = ['ticker', 'nome', 'preco', 'preco_justo_dcf', 'preco_justo_graham',
                'margem_seg_dcf_pct', 'margem_seg_graham_pct', 'undervalued', 'forte_desconto']
    display(valued_banks[val_cols].style.format({
        'preco': 'R$ {:.2f}',
        'preco_justo_dcf': 'R$ {:.2f}',
        'preco_justo_graham': 'R$ {:.2f}',
        'margem_seg_dcf_pct': '{:.1f}%',
        'margem_seg_graham_pct': '{:.1f}%',
    }))
else:
    valued_banks = pd.DataFrame()
    print("⚠️ Nenhum banco passou nos critérios de filtragem.")

Bancos identificados: 35
[filters] Bancos: 8/35 passaram nos critérios
[valuation] 5/8 bancos abaixo do preço justo | 5 com forte desconto (≥20%)


,ticker,nome,preco,preco_justo_dcf,preco_justo_graham,margem_seg_dcf_pct,margem_seg_graham_pct,undervalued,forte_desconto
0,ITSA4,ITAUSA PN EJ N1,R$ 13.31,R$ 10.36,R$ 14.23,-28.4%,6.5%,False,False
1,ABCB4,ABC BRASIL PN N2,R$ 24.71,R$ 31.19,R$ 34.96,20.8%,29.3%,True,True
2,BRSR6,BANRISUL PNB N1,R$ 16.37,R$ 31.73,R$ 30.94,48.4%,47.1%,True,True
3,WIZC3,WIZ CO ON NM,R$ 8.69,R$ 10.75,R$ 6.69,19.1%,-29.8%,False,False
4,ITSA3,ITAUSA ON EJ N1,R$ 13.18,R$ 10.36,R$ 14.23,-27.2%,7.4%,False,False
5,BPAC5,BTGP BANCO PNA N2,R$ 10.94,R$ 27.55,R$ 12.88,60.3%,15.1%,True,True
6,BRSR3,BANRISUL ON N1,R$ 18.18,R$ 31.73,R$ 30.94,42.7%,41.2%,True,True
7,BAZA3,AMAZONIA ON,R$ 82.20,R$ 141.87,R$ 148.86,42.1%,44.8%,True,True


## 5. 🏆 Top 20 — Ranking Final (Ações + Bancos)

Ações e bancos subvalorizados (preço < preço justo primário **e** Graham), ordenados por **margem de segurança média**.
⭐ = forte desconto (≥ 20% de margem de segurança).

In [ ]:
# Unir ações e bancos valorizados
all_valued = []
if len(valued_stocks) > 0:
    vs = valued_stocks.copy()
    vs['tipo'] = 'ação'
    all_valued.append(vs)
if len(valued_banks) > 0:
    vb = valued_banks.copy()
    vb['tipo'] = 'banco'
    all_valued.append(vb)

if all_valued:
    all_valued_df = pd.concat(all_valued, ignore_index=True)
else:
    all_valued_df = pd.DataFrame()

# Filtrar subvalorizados e rankear
if len(all_valued_df) > 0 and all_valued_df['undervalued'].any():
    undervalued = all_valued_df[all_valued_df['undervalued']].copy()

    # Ordenar por maior margem de segurança e pegar top 20
    top20 = (undervalued
             .sort_values('margem_seg_media_pct', ascending=False)
             .head(20)
             .reset_index(drop=True))

    print(f"🏆 TOP {len(top20)} SUBVALORIZADAS (ações + bancos)\n")

    result_cols = ['tipo', 'ticker', 'nome', 'setor', 'preco', 'preco_justo_dcf',
                   'preco_justo_graham', 'margem_seg_media_pct', 'forte_desconto',
                   'pl', 'pvp', 'roe_pct', 'margem_liquida_pct']
    display(top20[result_cols].style.format({
        'preco': 'R$ {:.2f}',
        'preco_justo_dcf': 'R$ {:.2f}',
        'preco_justo_graham': 'R$ {:.2f}',
        'margem_seg_media_pct': '{:.1f}%',
        'pl': '{:.2f}',
        'pvp': '{:.2f}',
        'roe_pct': '{:.1f}%',
        'margem_liquida_pct': '{:.1f}%',
    }).set_caption("Top 20 — Subvalorizadas por margem de segurança"))
else:
    top20 = pd.DataFrame()
    print("⚠️ Nenhuma ação/banco encontrado abaixo do preço justo.")

    if len(all_valued_df) > 0:
        print("\nMais próximas do preço justo:")
        near = all_valued_df.nlargest(10, 'margem_seg_media_pct')
        display(near[['tipo', 'ticker', 'nome', 'preco', 'preco_justo_dcf',
                       'preco_justo_graham', 'margem_seg_media_pct']])

🏆 TOP 20 SUBVALORIZADAS (ações + bancos)



,tipo,ticker,nome,setor,preco,preco_justo_dcf,preco_justo_graham,margem_seg_media_pct,forte_desconto,pl,pvp,roe_pct,margem_liquida_pct
0,ação,SAPR4,SANEPAR PN N2,Utilities,R$ 8.27,R$ 78.38,R$ 29.29,80.6%,True,6.41,0.20,17.9%,28.9%
1,ação,SAPR3,SANEPAR ON N2,Utilities,R$ 9.98,R$ 156.76,R$ 29.29,79.8%,True,7.74,0.24,17.9%,28.9%
2,ação,EALT4,ACO ALTONA PN,Industrials,R$ 13.23,R$ 108.91,R$ 35.05,75.1%,True,2.97,0.83,33.3%,18.3%
3,ação,ALUP4,ALUPAR PN N2,Utilities,R$ 10.81,R$ 53.28,R$ 23.69,67.0%,True,8.72,0.39,14.3%,27.6%
4,ação,VLID3,VALID ON NM,Industrials,R$ 20.10,R$ 81.46,R$ 35.34,59.2%,True,6.05,0.92,15.2%,12.7%
5,ação,RECV3,PETRORECSA ON NM,Energy,R$ 13.97,R$ 62.68,R$ 19.18,52.4%,True,6.41,0.90,13.6%,18.8%
6,ação,BLAU3,BLAU ON EJ NM,Healthcare,R$ 9.88,R$ 28.15,R$ 15.06,49.7%,True,7.72,1.00,13.5%,17.3%
7,ação,EUCA4,EUCATEX PN N1,Industrials,R$ 20.57,R$ 37.42,R$ 42.38,48.2%,True,5.95,0.68,12.0%,10.3%
8,banco,BRSR6,BANRISUL PNB N1,Financial Services,R$ 16.37,R$ 31.73,R$ 30.94,47.7%,True,3.91,0.58,15.7%,23.0%
9,banco,BAZA3,AMAZONIA ON,Financial Services,R$ 82.20,R$ 141.87,R$ 148.86,43.4%,True,3.93,0.63,15.2%,29.5%


## 7. Resumo da Análise

In [ ]:
# Sumário
print("=" * 70)
print("RESUMO DA ANÁLISE")
print("=" * 70)
print(f"Total de tickers analisados:     {len(tickers_df)}")
print(f"\nAções que passaram nos filtros:   {len(filtered_stocks)}")
if len(valued_stocks) > 0:
    print(f"  Subvalorizadas (ações):        {valued_stocks['undervalued'].sum()}")
    print(f"  Forte desconto (≥20%):         {valued_stocks['forte_desconto'].sum()}")
print(f"\nBancos que passaram nos filtros:  {len(filtered_banks)}")
if len(valued_banks) > 0:
    print(f"  Subvalorizados (bancos):       {valued_banks['undervalued'].sum()}")
    print(f"  Forte desconto (≥20%):         {valued_banks['forte_desconto'].sum()}")
print(f"\nTop 20 selecionadas:             {len(top20)}")
print("=" * 70)
print(f"\nModelos de valuation:")
print(f"  Ações: DCF 2-estágios (decay linear, {valuation.PROJECTION_YEARS}y) + DDM fallback + Graham")
print(f"  Bancos: Excess Returns + Graham")
print(f"\nParâmetros: Selic={valuation.SELIC*100:.2f}%, "
      f"Crescimento terminal={valuation.TERMINAL_GROWTH*100:.1f}%, "
      f"Projeção={valuation.PROJECTION_YEARS} anos")
print(f"Margem de segurança mínima (forte desconto): {valuation.MIN_SAFETY_MARGIN_PCT:.0f}%")

RESUMO DA ANÁLISE
Total de tickers analisados:     384

Ações que passaram nos filtros:   23
  Subvalorizadas (ações):        18
  Forte desconto (≥20%):         15

Bancos que passaram nos filtros:  8
  Subvalorizados (bancos):       5
  Forte desconto (≥20%):         5

Top 20 selecionadas:             20

Modelos de valuation:
  Ações: DCF 2-estágios (decay linear, 10y) + DDM fallback + Graham
  Bancos: Excess Returns + Graham

Parâmetros: Selic=14.25%, Crescimento terminal=3.5%, Projeção=10 anos
Margem de segurança mínima (forte desconto): 20%
